In [ ]:
# --- セル1: ライブラリのインポート ---
# !pip install captum torch scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import os
import matplotlib.pyplot as plt
import MeCab
import ipadic
import re

from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

# 日本語フォント設定
plt.rcParams['font.family'] = 'Meiryo'

print("ライブラリの準備完了")

ライブラリの準備完了


In [2]:
# --- セル2: トークナイザ定義とモデル読み込み ---

# ==========================================
# ★設定: 分析したいジャンルID
TARGET_GENRE_ID = 201
# ==========================================

# ジャンル対応表
genres_map = {
    0: '未設定',
    101: '異世界（恋愛）', 102: '現実世界（恋愛）',
    201: 'ハイファンタジー', 202: 'ローファンタジー',
    301: '純文学', 302: 'ヒューマンドラマ', 303: '歴史',
    304: '推理', 305: 'ホラー', 306: 'アクション', 307: 'コメディー',
    401: 'VRゲーム', 402: '宇宙', 403: '空想科学', 404: 'パニック',
    9901: '童話', 9902: '詩', 9903: 'エッセイ', 9904: 'リプレイ',
    9999: 'その他', 9801: 'ノンジャンル'
}

# ジャンル名の取得
genre_name = genres_map.get(TARGET_GENRE_ID, 'all')
print(f"Target Genre: {genre_name}")

# --- 1. トークナイザの再定義 (URL処理を追加) ---
chaser = MeCab.Tagger(ipadic.MECAB_ARGS)

def japanese_tokenizer(text):
    text = str(text)
    
    # ★追加: URLを <URL> に置換する処理
    # これにより、分析時も学習時と同じ状態でテキストが処理されます
    text = re.sub(r'https?://[\w/:%#\$&\?\(\)~\.=\+\-]+', '<URL>', text)
    
    parsed = chaser.parse(text)
    words = []
    for line in parsed.split('\n'):
        if line == 'EOS' or line == '': continue
        parts = line.split('\t')
        word_surface = parts[0]
        features = parts[1].split(',')
        pos = features[0]
        target_pos = ['名詞', '動詞', '形容詞', '接頭詞']
        if pos in target_pos:
            if features[1] in ['非自立', '代名詞', '数']: continue
            if len(features) > 6 and features[6] != '*':
                word = features[6]
            else:
                word = word_surface
            words.append(word)
    return words

# --- 2. モデルのロード ---
model_path = f'result_tfidf/model_{genre_name}.pkl'

if os.path.exists(model_path):
    loaded_pipeline = joblib.load(model_path)
    print(f"モデル読み込み成功: {model_path}")
    tfidf_vec = loaded_pipeline.named_steps['tfidf']
    lr_model = loaded_pipeline.named_steps['clf']
else:
    raise FileNotFoundError(f"モデルファイルが見つかりません: {model_path}")

Target Genre: ハイファンタジー
モデル読み込み成功: result_tfidf/model_ハイファンタジー.pkl


c:\Users\blast\miniconda3\envs\lime_env\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\blast\miniconda3\envs\lime_env\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\blast\miniconda3\envs\lime_env\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 w

In [3]:
# --- セル3: Scikit-learnモデルをPyTorchモデルへ変換 ---

class PyTorchLogisticRegression(nn.Module):
    def __init__(self, sklearn_model):
        super().__init__()
        weights = torch.tensor(sklearn_model.coef_, dtype=torch.float32)
        bias = torch.tensor(sklearn_model.intercept_, dtype=torch.float32)
        self.input_dim = weights.shape[1]
        self.linear = nn.Linear(self.input_dim, 1)
        with torch.no_grad():
            self.linear.weight.copy_(weights)
            self.linear.bias.copy_(bias)
            
    def forward(self, x):
        return self.linear(x)

torch_model = PyTorchLogisticRegression(lr_model)
torch_model.eval()
print("PyTorchモデルへの変換完了")

PyTorchモデルへの変換完了


In [7]:
# --- セル4: 分析データの選択 ---
csv_path = 'dataset/narou_dataset.csv'
df = pd.read_csv(csv_path)

df['あらすじ'] = df['あらすじ'].fillna('')
df['タイトル'] = df['タイトル'].fillna('無題')

if TARGET_GENRE_ID is not None:
    df_genre = df[df['作品ジャンル'] == TARGET_GENRE_ID]
else:
    df_genre = df

if len(df_genre) == 0:
    raise ValueError("該当するジャンルのデータがありません")

# ランダムに1件抽出
target_row = df_genre.sample(n=1).iloc[0]

target_text = target_row['あらすじ']
target_title = target_row['タイトル']
true_label = target_row['is_eternal']

print(f"【分析対象】")
print(f"タイトル: {target_title}")
print(f"正解: {'エタる (1)' if true_label==1 else '完結 (0)'}")
print("-" * 20)
print(target_text)

【分析対象】
タイトル: BEYOND SOUL
正解: 完結 (0)
--------------------
時は、中世・唐の時代。自身の思想を&quot;思超&quot;と呼ばれる超能力化させた&quot;思超家&quot;たちが世界中に住んでいた。洛陽に住む思超家の孫・司馬章は、安史の乱に巻き込まれ、天上王真理公子(天理)を名乗る謎の男に祖父を殺されてしまう。やがて祖父の思超を受け継いだ司馬章は、祖父の仇である天理を倒すためにアジア全体を股にかけた大冒険を始める。


In [8]:
# --- セル5: Captum (Integrated Gradients) の実行 (表示改良版) ---

print("Captumによる分析を開始します...")

# 1. TF-IDF変換
tfidf_vector_sparse = tfidf_vec.transform([target_text])
input_tensor = torch.tensor(tfidf_vector_sparse.toarray(), dtype=torch.float32)
input_tensor.requires_grad = True

# 2. Integrated Gradients
ig = IntegratedGradients(torch_model)
attributions, delta = ig.attribute(
    inputs=input_tensor,
    baselines=torch.zeros_like(input_tensor),
    target=0, 
    return_convergence_delta=True
)

attributions = attributions.detach().numpy()[0]
input_values = input_tensor.detach().numpy()[0]

# 3. 結果の整理
feature_names = np.array(tfidf_vec.get_feature_names_out())
non_zero_indices = input_values > 0

# 元の単語リスト
active_words = feature_names[non_zero_indices]
active_attrs = attributions[non_zero_indices]

# ===========================================================
# ★追加: N-gramのスペースを可視化用に置換する処理
# 例: "拉致 する" -> "拉致_する" に変換して、つながっていることを強調します
# 好きな記号に変えてください ('_' や '・' や '+' など)
active_words_display = [word.replace(' ', '_') for word in active_words]
# ===========================================================

# 予測スコア
with torch.no_grad():
    logits = torch_model(input_tensor).item()
    prob = torch.sigmoid(torch.tensor(logits)).item()

pred_label = 1 if prob > 0.5 else 0

print(f"\n【AI判定結果】")
print(f"Logits: {logits:.4f}")
print(f"エタる確率: {prob*100:.2f}%")
print(f"判定: {'エタる (1)' if pred_label==1 else '完結 (0)'}")

# VisualizationDataRecord作成
# raw_input_ids に加工後の active_words_display を渡します
viz_record = viz.VisualizationDataRecord(
    word_attributions=active_attrs,
    pred_prob=prob,
    pred_class=pred_label,
    true_class=true_label,
    attr_class="エタる" if pred_label==1 else "完結",
    attr_score=np.sum(active_attrs),
    raw_input_ids=active_words_display, # ★ここを変更
    convergence_score=delta
)

print("\n分析完了。")

Captumによる分析を開始します...

【AI判定結果】
Logits: -0.3495
エタる確率: 41.35%
判定: 完結 (0)

分析完了。


In [9]:
# --- セル6: 結果の可視化 ---

print(f"【タイトル】: {target_title}")
print("赤色: 「エタる」と判断させた単語 (プラスの影響)")
print("青色: 「完結」と判断させた単語 (マイナスの影響)")
print("※TF-IDF分析のため、元の文章順ではなく、抽出されたキーワードとして表示されます。\n")

_ = viz.visualize_text([viz_record])

【タイトル】: BEYOND SOUL
赤色: 「エタる」と判断させた単語 (プラスの影響)
青色: 「完結」と判断させた単語 (マイナスの影響)
※TF-IDF分析のため、元の文章順ではなく、抽出されたキーワードとして表示されます。

